In [ ]:
%pip install scikit-learn
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


^C
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import xgboost as xgb
%pip install matplotlib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,classification_report
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_sample_weight


In [ ]:
%pip install pandas numpy

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
edstays = pd.read_csv("data/mimic-iv-ed-2.2/ed/edstays.csv.gz")
triage = pd.read_csv("data/mimic-iv-ed-2.2/ed/triage.csv.gz")
vitals = pd.read_csv("data/mimic-iv-ed-2.2/ed/vitalsign.csv.gz")
medrecon = pd.read_csv("data/mimic-iv-ed-2.2/ed/medrecon.csv.gz")
pyxis = pd.read_csv("data/mimic-iv-ed-2.2/ed/pyxis.csv.gz")
diagnosis = pd.read_csv("data/mimic-iv-ed-2.2/ed/diagnosis.csv.gz")
erha = pd.read_csv("data/mimic-iv-ed-2.2/ed/complaint_to_erha_llm.csv")

In [ ]:
print(edstays.columns)
print(triage.columns)
print(vitals.columns)
print(diagnosis.columns)
print(pyxis.columns)
print(medrecon.columns)
print(erha.columns)


In [ ]:
ed_triage = edstays.merge(triage, on="stay_id", how="left")

In [ ]:
vitals_first = (vitals.sort_values("charttime").groupby("stay_id").first().reset_index())

In [ ]:
ed_triage = ed_triage.merge(
    vitals_first[["stay_id", "charttime", "rhythm"]],
    on="stay_id",
    how="left"
)

In [ ]:
ed_triage["Chief_Complaints"] = (ed_triage["chiefcomplaint"].str.strip().str.lower().str.replace(r"\s+", " ", regex=True))
erha["Chief_Complaints"] = (erha["complaint"].str.strip().str.lower().str.replace(r"\s+", " ", regex=True))
triage["Chief_Complaints"] = (triage["chiefcomplaint"].str.strip().str.lower().str.replace(r"\s+", " ", regex=True))
print(erha.columns)
print(ed_triage.columns)

In [ ]:
print(ed_triage.shape)
print(erha.shape)
ed_triage = ed_triage.merge(
    erha[["Chief_Complaints", "erha_codes", "erha_names"]],
    on="Chief_Complaints",
    how="left"
)

In [ ]:
ed_triage.tail(2)

In [ ]:
print(ed_triage["Chief_Complaints"].nunique())
print(erha["complaint"].nunique())

In [ ]:
ed_triage.isnull().sum().sort_values(ascending=False)

In [ ]:
ed_triage.dtypes

In [ ]:
ed_triage['pain'] = pd.to_numeric(ed_triage['pain'],errors='coerce')
ed_triage['rhythm'] = pd.to_numeric(ed_triage['rhythm'],errors='coerce')

In [ ]:
ed_triage["acuity_group"] = np.nan

#ed_triage.loc[ed_triage["acuity"] == 1, "acuity_group"] = 0
#ed_triage.loc[ed_triage["acuity"] == 2, "acuity_group"] = 1
#ed_triage.loc[ed_triage["acuity"] == 3, "acuity_group"] = 2
#ed_triage.loc[ed_triage["acuity"] == 4, "acuity_group"] = 3
#ed_triage.loc[ed_triage["acuity"] == 5, "acuity_group"] = 4

ed_triage.loc[ed_triage["acuity"] < 3, "acuity_group"] = 0
ed_triage.loc[ed_triage["acuity"] == 3, "acuity_group"] = 1
ed_triage.loc[ed_triage["acuity"] > 3, "acuity_group"] = 2

ed_triage = ed_triage.dropna(subset=["acuity_group"])

In [ ]:
ed_triage["stay_id"].nunique()

In [ ]:
ed_triage["acuity"].value_counts(dropna=False)

In [ ]:
numeric_features = [
    'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain',]
categorical_features = ['arrival_transport','race', 'gender','erha_names','erha_codes']
feature =numeric_features+ categorical_features  + ["Chief_Complaints"]
x = ed_triage[feature]
y = ed_triage['acuity_group']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42,stratify=y)

In [ ]:
print(y_train.value_counts())

In [ ]:
x_train = x_train.copy()
x_test = x_test.copy()

x_train["Chief_Complaints"] = (
    x_train["Chief_Complaints"]
    .fillna("")
    .astype(str)
)

x_test["Chief_Complaints"] = (
    x_test["Chief_Complaints"]
    .fillna("")
    .astype(str)
)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('str', categorical_pipeline, categorical_features),
     ('text', TfidfVectorizer(max_features=500,ngram_range=(1, 2), stop_words="english"), "Chief_Complaints")
])
    

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
    n_estimators=700,
    learning_rate=0.01,
    max_depth=5
    ))
])


model.fit(x_train, y_train)

predictions = model.predict(x_test)

In [ ]:
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(
    y_test,
    predictions,
    average="weighted"
)

recall = recall_score(
    y_test,
    predictions,
    average="weighted"
)

f1 = f1_score(
    y_test,
    predictions,
    average="weighted"
)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

In [ ]:
print(classification_report(
    y_test,
    predictions,
    target_names=["high","mid","low"]
))

In [ ]:
cm = confusion_matrix(y_test,predictions)
cfd = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["high","mid","low"]
)
cfd.plot()
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.show()

In [ ]:
model.named_steps["classifier"].feature_importances_